# Automotive Requirements Analyzer and Acceptance Test Generator

This notebook demonstrates a multi-agent system which analyzes and validates documents under requirements/weather-app/business for consistency and always generates user acceptance tests in MD file format. The multi-agent system is implemented using Strands agents deployed on Amazon Bedrock AgentCore Runtime with inbound authentication.

## Overview

The system implements a graph multi-agent pattern with 2 steps:
1. **Requirements Consistency Analyzer**: Analyzes business requirements documents for consistency
2. **User Acceptance Test Generator**: If no severe violations are found, generates comprehensive user acceptance tests

### Architecture

- **Agent 1**: Requirements Consistency Analyzer
- **Agent 2**: User Acceptance Test Generator
- **Authentication**: Amazon Cognito with JWT tokens
- **Runtime**: Amazon Bedrock AgentCore Runtime
- **Model**: Amazon Nova Lite 2

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Multi-agent automotive analysis                                                  |
| Agent type          | Graph pattern with 2 agents                                                     |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Amazon Nova Lite 2                                                     |
| Tutorial components | Requirements analysis, User acceptance test generation, AgentCore Runtime      |
| Tutorial vertical   | Automotive                                                                       |
| Example complexity  | Intermediate                                                                     |
| Inbound Auth        | Cognito                                                                          |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                    |

## Prerequisites

To execute this tutorial you will need:
* Python 3.13+ (Recommended: virtual environment at workspace level created and top level requirements installed)
* AWS credentials configured for a US based region

In [ ]:
# Optionally set your AWS_PROFILE and AWS_REGION (region has to be US based)
import os
# os.environ['AWS_PROFILE'] = 'default'
# os.environ['AWS_REGION'] = 'us-west-2'

In [ ]:
# Optionally check that required packages are installed
!pip install --force-reinstall -r ../requirements-agent/requirements.txt --quiet

## Setting up Amazon Cognito for Authentication

Let's provision a Cognito User Pool with an App client and one test user for automotive applications.

In [ ]:
import sys
import os
import time

# Import our utility functions
from utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool for automotive applications...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed")

## Preparing the Automotive Requirements Analyzer Agent Graph

Let's examine our multi-agent graph system that implements requirements consistency checking and user acceptance test generation using the Strands GraphBuilder pattern.

### Graph Architecture

Our system uses a **conditional sequential graph** pattern:

```
┌─────────────────────────┐                         ┌─────────────────────────────┐
│ Requirements Consistency│────-------------------─▶│ User Acceptance Test        │
│ Analyzer (Entry Point)  │                         │ Generator (Conditional Node)│
└─────────────────────────┘                         └─────────────────────────────┘
```

- **Node 1**: Requirements Consistency Analyzer analyzes business requirements documents for consistency, completeness, and quality issues
- **Node 2**: User Acceptance Test Generator creates comprehensive test specifications

This ensures that user acceptance tests are only generated for business requirements that meet automotive quality standards.

## Loading Business Requirements Documents

Let's load the weather application business requirements documents from the sample-data/weather-app folder that we'll analyze.

In [ ]:
import os
import glob

# Load all business requirements documents from the sample data folder
business_folder = '../sample-data/weather-app'
business_documents = {}
combined_requirements = ""

# Find all markdown files starting with weather_app_brd
business_files = glob.glob(os.path.join(business_folder, 'weather_app_brd*.md'))

print(f"Found {len(business_files)} business requirements documents:")
for file_path in business_files:
    filename = os.path.basename(file_path)
    print(f"  - {filename}")
    
    with open(file_path, 'r') as f:
        content = f.read()
        business_documents[filename] = content
        
        # Add to combined requirements
        combined_requirements += f"\n\n# {filename}\n{content}\n"

print("\nBusiness requirements documents loaded successfully!")
for filename, content in business_documents.items():
    print(f"{filename}: {len(content)} characters")
print(f"\nTotal combined requirements: {len(combined_requirements)} characters")

## Deploying the Agent to AgentCore Runtime

Now we'll configure and deploy our automotive requirements analyzer to AgentCore Runtime with inbound authentication.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url")
client_id = cognito_config.get("client_id")

print(f"Discovery URL: {discovery_url}")
print(f"Client ID: {client_id}")

### Configure AgentCore Runtime Deployment

In [ ]:
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="../requirements-agent/requirements_analyzer.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="../requirements-agent/requirements.txt",
    region=region,
    agent_name="requirements_analyzer_nova_2_lite",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
print("Configuration response:", response)

### Launch Agent to AgentCore Runtime

In [ ]:
print("Launching automotive requirements analyzer to AgentCore Runtime...")
launch_result = agentcore_runtime.launch()
print("Launch result:", launch_result)

### Check AgentCore Runtime Status

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print(f"Initial status: {status}")
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

print(f"Final status: {status}")

## Testing the Automotive Requirements Analyzer

Let's test our multi-agent system with the weather application requirements documents.

In [ ]:
import re
from IPython.display import display, Markdown

def parse_agent_response(response_str):
    """Parse agent response and extract markdown content."""
    
    def extract_text(text):
        text = text.replace('\\\\n', '\n')
        text = text.replace("\\\\'" , "'")
        text = text.replace('\\\\"', '"')
        return text
    
    match = re.search(r"'text': '(.*?)'\}]}", response_str, re.DOTALL)
    if match:
        return extract_text(match.group(1))
    return "Could not parse response"

### Test Case 1: Complete Requirements Analysis

In [ ]:
print("Business requirements documents loaded:")
print("=" * 50)
for filename, content in business_documents.items():
    print(f"{filename}: {len(content)} characters")
print(f"Total combined: {len(combined_requirements)} characters")
print("=" * 50)

# Get fresh bearer token
bearer_token = reauthenticate_user(cognito_config.get("client_id"))

# Test with business requirements documents
print("\nTesting requirements analysis with weather application business documents...")
invoke_response = agentcore_runtime.invoke(
    {"requirements_docs": combined_requirements}, 
    bearer_token=bearer_token, 
)
print("Response:", invoke_response)
response_text = parse_agent_response(invoke_response['response'])
display(Markdown(response_text))

### Test Case 2: Incomplete Requirements

In [ ]:
# Create incomplete requirements with missing critical sections
incomplete_requirements = """
# Incomplete Weather Application Requirements

## Basic Description
A weather application for vehicles.

## Some Features
- Show weather
- Maybe voice commands
- Display temperature

## Missing Sections
- No detailed functional requirements
- No non-functional requirements
- No security requirements
- No performance criteria
- No acceptance criteria
"""

print("Testing with incomplete requirements (should identify severe issues but still generate UATs)...")
invoke_response = agentcore_runtime.invoke(
    {"requirements_docs": incomplete_requirements}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)
# Parse and display
response_text = parse_agent_response(invoke_response['response'])
display(Markdown(response_text))

### Test Case 3: Requirements with Conflicting Information

In [ ]:
# Create requirements with conflicts
conflicting_requirements = """
# Conflicting Weather Application Requirements

## Business Requirements
- The application must load within 2 seconds
- The application should support offline mode
- Budget is limited to $5,000

## Technical Requirements  
- The application must load within 5 seconds (CONFLICT with business req)
- The application requires constant internet connectivity (CONFLICT with offline mode)
- Implementation requires 6 months with 5 developers (CONFLICT with budget)

## Additional Conflicts
- Business says "simple interface" but technical requires "complex multi-screen UI"
- Business says "Android Auto only" but technical mentions "iOS CarPlay support"
"""

print("Testing with conflicting requirements (should identify consistency issues but still generate UATs)...")
invoke_response = agentcore_runtime.invoke(
    {"requirements_docs": conflicting_requirements}, 
    bearer_token=bearer_token
)
print("Response:", invoke_response)
# Parse and display
response_text = parse_agent_response(invoke_response['response'])
display(Markdown(response_text))

## Testing Without Authorization (Should Fail)

In [ ]:
# This should fail with AccessDeniedException
try:
    print("Testing without authorization (should fail)...")
    invoke_response = agentcore_runtime.invoke({"requirements_docs": combined_requirements})
    print("Unexpected success:", invoke_response)
except Exception as e:
    print(f"Expected error: {e}")

## Cleanup (Optional)

Clean up the AgentCore Runtime and ECR repository when done testing.

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# import boto3

# agentcore_control_client = boto3.client(
#     'bedrock-agentcore-control',
#     region_name=region
# )
# ecr_client = boto3.client(
#     'ecr',
#     region_name=region
# )

# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,
# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1].split(':')[0],
#     force=True
# )

# print("Cleanup Done.")

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# import os

# if os.path.exists('.bedrock_agentcore.yaml'):
#     os.remove('.bedrock_agentcore.yaml')
#     print("Deleted .bedrock_agentcore.yaml file")

# print("Cleanup of local files done.")

## Summary

This notebook demonstrated:

1. **Multi-Agent Graph Architecture**: Implemented using Strands GraphBuilder with conditional execution
   - Requirements Consistency Analyzer (Entry Node)
   - User Acceptance Test Generator (Conditional Node)
   - Conditional edge based on analysis severity

2. **Conditional Workflow**: User acceptance tests are only generated if no severe consistency issues are found
   - Graph execution metrics and node tracking
   - Automatic workflow control based on analysis results

3. **Automotive Focus**: Specialized for automotive requirements analysis with quality standards
   - Requirements consistency checking
   - Completeness and quality validation
   - Safety-critical requirements verification

4. **Security**: Implemented inbound authentication using Amazon Cognito
   - JWT token validation
   - Secure multi-agent execution

5. **Scalability**: Deployed on Amazon Bedrock AgentCore Runtime for production use
   - Graph-based agent orchestration
   - Performance monitoring and metrics

The system provides a comprehensive solution for automotive business requirements analysis, using advanced multi-agent graph patterns to analyze business requirements quality and consistency while always generating user acceptance tests for complete test coverage.